# §11.2.5 — 입력 기울기로 유효 수용장 실측하기

> 딥러닝 교재 · 3부 11장 2절 5항 (🐍)
> 선행: §11.2.1(재귀 공식) · §11.2.2(팽창) · §11.2.3(유효 수용장의 가우시안 형태)

## 이 노트북이 답하는 질문

1. **§11.2.1의 공식이 맞는가?** 기울기가 0이 아닌 범위가 정확히 이론 수용장인지 센다.
2. **§11.2.3의 스케치가 맞는가?** 기여 분포가 종 모양이 되고 유효 폭이 $\sqrt{L}$로 자라는가.
3. **같은 유효 커널 크기라도 팽창 스케줄에 따라** 기여가 닿는 위치가 달라지는가 (§11.2.6의 격자 인공물).
4. **학습은 유효 수용장을 바꾸는가?** 장거리 과제를 학습시켜 전후를 비교한다.

**예상 실행 시간** CPU 약 90초 (`FAST = True`이면 약 30초).
이 실험을 이해하려면 §11.2.1의 유도와 §11.2.3의 무작위 행보 논증이 필요합니다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 1차원 합성곱층과 기울기 — 원리 그대로 구현

유효 수용장은 정의상 **기울기** $\partial y_{\text{center}}/\partial x$다 (§11.2.5).
순전파와 역전파를 팽창까지 포함해 직접 구현한다. 채널이 있는 1차원 합성곱은
$y[c',t]=\sum_c\sum_{j} x[c,\,t+d(j-\lfloor k/2\rfloor)]\,w[c',c,j]$ (same 패딩).

In [ ]:
def conv1d_fwd(x, W, d=1):
    # x: (C,T), W: (Cout,C,k) -> y: (Cout,T), 제로 same 패딩, 팽창 d
    C, T = x.shape; Co, _, k = W.shape
    h = (k // 2) * d
    xp = np.zeros((C, T + 2*h)); xp[:, h:h+T] = x
    cols = np.stack([xp[:, j*d : j*d + T] for j in range(k)], axis=2)  # (C,T,k)
    y = np.einsum('ctk,ock->ot', cols, W, optimize=True)
    return y, cols

def conv1d_bwd_x(dy, W, T, d=1):
    # dy: (Cout,T) -> dx: (C,T)
    Co, C, k = W.shape
    h = (k // 2) * d
    dxp = np.zeros((C, T + 2*h))
    for j in range(k):
        dxp[:, j*d : j*d + T] += np.einsum('ot,ock->ct', dy, W[:, :, j:j+1], optimize=True)
    return dxp[:, h:h+T]

def conv1d_bwd_W(dy, cols):
    return np.einsum('ot,ctk->ock', dy, cols, optimize=True)

class Net1D:
    # conv 스택 (+ReLU) -> 중앙 위치의 채널 평균을 스칼라 출력으로
    def __init__(self, C, depths_d, k=3, rn=None, relu=True):
        rn = rn or np.random.default_rng(0)
        self.d = list(depths_d); self.k = k; self.relu = relu
        self.Ws = []
        cin = 1
        for _ in self.d:
            self.Ws.append(rn.standard_normal((C, cin, k)) * np.sqrt(2/(cin*k)))
            cin = C
        self.u = rn.standard_normal(C) / np.sqrt(C)
    def forward(self, x):
        a = x[None, :]                       # (1,T)
        self.caches = []
        for W, d in zip(self.Ws, self.d):
            z, cols = conv1d_fwd(a, W, d)
            m = (z > 0) if self.relu else np.ones_like(z, bool)
            a = z * m
            self.caches.append((cols, m, a.shape[1]))
        T = a.shape[1]
        self.center = T // 2
        out = self.u @ a[:, self.center]
        self.a_last = a
        return out
    def grad_x(self, T):
        # dy/dx : 중앙 스칼라 출력에 대한 입력 기울기
        da = np.zeros_like(self.a_last)
        da[:, self.center] = self.u
        for (cols, m, Tl), W, d in zip(reversed(self.caches), reversed(self.Ws), reversed(self.d)):
            dz = da * m
            da = conv1d_bwd_x(dz, W, Tl, d)
        return da[0]
    def grad_all(self, x):
        self.forward(x)
        # 파라미터 기울기 (학습용)
        da = np.zeros_like(self.a_last)
        da[:, self.center] = self.u
        du = self.a_last[:, self.center].copy()
        gWs = [None]*len(self.Ws)
        for i in range(len(self.Ws)-1, -1, -1):
            cols, m, Tl = self.caches[i]
            dz = da * m
            gWs[i] = conv1d_bwd_W(dz, cols)
            da = conv1d_bwd_x(dz, self.Ws[i], Tl, self.d[i])
        return gWs, du

T = 129
x0 = rng.standard_normal(T)
net = Net1D(8, [1, 1, 1], rn=np.random.default_rng(1))
_ = net.forward(x0)
g = net.grad_x(T)
nz = np.nonzero(np.abs(g) > 1e-12)[0]
print(f"깊이 3, k=3: 기울기 비영 범위 = {nz.max()-nz.min()+1} (이론 r_L = {1+2*3})")

> 이론 수용장은 **기울기가 0이 아닐 수 있는 최대 범위**로 즉시 검산된다. 이하 모든 실측이 이 검산 위에서 진행된다.

---
## 2. 기여 분포의 모양 — 깊이를 훑는다

무작위 초기화 여러 개에 대해 $|\partial y/\partial x_j|$의 평균을 상대 위치 $j$의 함수로 그린다.
§11.2.3의 예측: 종 모양, 유효 폭 $\propto \sqrt{L}$.

In [ ]:
DEPTHS = [2, 5, 10, 20]
N_INIT = 10 if FAST else 30
C = 8
profiles = {}
for L in DEPTHS:
    acc_g = np.zeros(T)
    for i in range(N_INIT):
        net = Net1D(C, [1]*L, rn=np.random.default_rng(100 + i))
        net.forward(rng.standard_normal(T))
        acc_g += np.abs(net.grad_x(T))
    profiles[L] = acc_g / N_INIT
print("완료", {L: round(profiles[L].max(), 3) for L in DEPTHS})

In [ ]:
def eff_width(p):
    # 2차 모멘트 기반 유효 폭 (±2σ)
    j = np.arange(len(p)) - len(p)//2
    w = p / p.sum()
    var = np.sum(w * j**2) - np.sum(w * j)**2
    return 4*np.sqrt(var), np.sqrt(var)

LS = list(range(1, 21, 1 if not FAST else 2))
theo, meas = [], []
for L in LS:
    accg = np.zeros(T)
    for i in range(6):
        net = Net1D(C, [1]*L, rn=np.random.default_rng(300 + 17*i + L))
        net.forward(rng.standard_normal(T))
        accg += np.abs(net.grad_x(T))
    theo.append(1 + 2*L)
    meas.append(eff_width(accg)[0])
print("이론:", theo[:5], "... 실측(±2σ):", np.round(meas[:5], 1), "...")

---
## 3. 팽창 스케줄 — 같은 도달 범위, 다른 도달 위치

스케줄 $[1,2,4]$와 $[2,2,2]$는 이론 수용장이 비슷하지만, $[2,2,2]$는 짝수 위치만 닿는다 (§11.2.6).

In [ ]:
sched_a, sched_b = [1, 2, 4], [2, 2, 2]
ga = np.zeros(T); gb = np.zeros(T)
for i in range(N_INIT):
    na = Net1D(C, sched_a, rn=np.random.default_rng(500 + i)); na.forward(rng.standard_normal(T)); ga += np.abs(na.grad_x(T))
    nb = Net1D(C, sched_b, rn=np.random.default_rng(700 + i)); nb.forward(rng.standard_normal(T)); gb += np.abs(nb.grad_x(T))
ga /= N_INIT; gb /= N_INIT
za = np.sum(gb[1::2] > 1e-12)
print(f"[2,2,2] 스케줄에서 홀수 위치의 비영 기여 개수: {za} (격자 인공물)")

---
## 4. 학습이 유효 수용장을 바꾸는가 — 장거리 과제

$y=\operatorname{sign}(x[c-\Delta]+x[c+\Delta])$, $\Delta=12$: 중앙에서 $\pm12$ 떨어진 **두 값만**이 답을 정한다.
중앙에 몰린 초기 기여로는 풀 수 없고, **학습이 기여 질량을 $\pm\Delta$로 옮겨야만** 풀린다.

In [ ]:
DELTA = 12
def make_lr_data(n, rn):
    X = rn.standard_normal((n, T))
    c = T // 2
    y = np.sign(X[:, c-DELTA] + X[:, c+DELTA])
    return X, y

net_lr = Net1D(12, [1, 2, 4, 8], rn=np.random.default_rng(11))
g_before = None
Xtr, ytr = make_lr_data(1500, np.random.default_rng(SEED))
Xev, yev = make_lr_data(1500, np.random.default_rng(SEED + 1))

# 학습 전 ERF
gacc = np.zeros(T)
for i in range(60):
    net_lr.forward(Xtr[i]); gacc += np.abs(net_lr.grad_x(T))
g_before = gacc / 60

STEPS = 150 if FAST else 400
ms = [np.zeros_like(W) for W in net_lr.Ws]; vs = [np.zeros_like(W) for W in net_lr.Ws]
mu = np.zeros_like(net_lr.u); vu = np.zeros_like(net_lr.u)
B = 32
rb = np.random.default_rng(2)
for t in range(1, STEPS+1):
    idx = rb.integers(0, len(ytr), B)
    gW_sum = [np.zeros_like(W) for W in net_lr.Ws]; gu_sum = np.zeros_like(net_lr.u)
    for ii in idx:
        out = net_lr.forward(Xtr[ii])
        p = 1/(1+np.exp(-np.clip(out, -30, 30)))
        dz = (p - (ytr[ii] > 0))
        gWs, du = net_lr.grad_all(Xtr[ii])
        for a_, b_ in zip(gW_sum, gWs): a_ += dz * b_
        gu_sum += dz * du
    for W, gW, m, v in zip(net_lr.Ws, gW_sum, ms, vs):
        adamW = gW / B
        m[:] = 0.9*m + 0.1*adamW; v[:] = 0.999*v + 0.001*adamW**2
        W -= 6e-3 * (m/(1-0.9**t)) / (np.sqrt(v/(1-0.999**t)) + 1e-8)
    mu[:] = 0.9*mu + 0.1*(gu_sum/B); vu[:] = 0.999*vu + 0.001*(gu_sum/B)**2
    net_lr.u -= 6e-3 * (mu/(1-0.9**t)) / (np.sqrt(vu/(1-0.999**t)) + 1e-8)

hits = 0
for ii in range(len(yev)):
    hits += (net_lr.forward(Xev[ii]) > 0) == (yev[ii] > 0)
print(f"장거리 과제 시험 정확도: {hits/len(yev):.3f}")

gacc = np.zeros(T)
for i in range(60):
    net_lr.forward(Xtr[i]); gacc += np.abs(net_lr.grad_x(T))
g_after = gacc / 60

---
## 5. 교재 그림 — fig_11_2_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()
j = np.arange(T) - T//2

# (a) 깊이별 기여 분포 + 가우시안 근사
ax = axes[0]
for i, L in enumerate(DEPTHS):
    p = profiles[L] / profiles[L].max()
    ax.plot(j, p, color=CB[i+1], lw=1.4, label=lab(f'깊이 {L}', f'depth {L}'))
    _, sg = eff_width(profiles[L])
    ax.plot(j, np.exp(-j**2/(2*sg**2)), color=CB[i+1], lw=0.8, ls='--', alpha=0.6)
ax.set_xlim(-40, 40)
ax.set_xlabel(lab('입력 상대 위치', 'input offset'))
ax.set_ylabel(lab('정규화된 $|\\partial y/\\partial x_j|$', 'normalized gradient'))
ax.set_title(lab('(a) 기여 분포와 가우시안 근사(점선)', '(a) contribution profile vs Gaussian'), fontsize=10)
ax.legend(fontsize=8)

# (b) 이론 vs 유효 수용장
ax = axes[1]
ax.plot(LS, theo, 'o-', color=CB[0], ms=3, label=lab('이론 $r_L=1+2L$', 'theoretical'))
ax.plot(LS, meas, 's-', color=CB[5], ms=3, label=lab('유효 폭 ($\\pm2\\sigma$ 실측)', 'effective (measured)'))
Lf = np.array(LS, float)
ax.plot(LS, meas[0]*np.sqrt(Lf/Lf[0]), ':', color=CB[5], lw=1,
        label=lab('$\\sqrt{L}$ 안내선', '$\\sqrt{L}$ guide'))
ax.set_xlabel(lab('깊이 $L$ (층)', 'depth $L$'))
ax.set_ylabel(lab('폭 (픽셀)', 'width (pixels)'))
ax.set_title(lab('(b) 이론 수용장은 선형, 유효 수용장은 $\\sqrt{L}$', '(b) theoretical vs effective RF'), fontsize=10)
ax.legend(fontsize=8)

# (c) 팽창 스케줄
ax = axes[2]
ax.stem(j, ga/ga.max(), linefmt=CB[5], markerfmt=' ', basefmt=' ',
        label=lab('팽창 $[1,2,4]$', 'dilation $[1,2,4]$'))
ax.stem(j, -gb/gb.max(), linefmt=CB[4], markerfmt=' ', basefmt=' ',
        label=lab('팽창 $[2,2,2]$ (아래로 표시)', 'dilation $[2,2,2]$'))
ax.axhline(0, color='k', lw=0.6)
ax.set_xlim(-18, 18)
ax.set_xlabel(lab('입력 상대 위치', 'input offset'))
ax.set_ylabel(lab('정규화 기여 (부호는 표시용)', 'normalized contribution'))
ax.set_title(lab('(c) $[2,2,2]$는 짝수 위치만 닿는다 — 격자 인공물', '(c) gridding with equal dilations'), fontsize=10)
ax.legend(fontsize=8)

# (d) 학습 전후
ax = axes[3]
ax.plot(j, g_before/g_before.max(), color=CB[0], lw=1.2, label=lab('학습 전', 'before training'))
ax.plot(j, g_after/g_after.max(), color=CB[3], lw=1.4, label=lab('학습 후', 'after training'))
for dd in (-DELTA, DELTA):
    ax.axvline(dd, color=CB[4], lw=0.8, ls=':')
ax.set_xlim(-32, 32)
ax.set_xlabel(lab('입력 상대 위치', 'input offset'))
ax.set_ylabel(lab('정규화된 $|\\partial y/\\partial x_j|$', 'normalized gradient'))
ax.set_title(lab(f'(d) 장거리 과제($\\pm{DELTA}$) 학습이 기여를 옮긴다', '(d) training reshapes the ERF'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_11_2_5')
plt.show()

> ### 읽는 법
>
> (a) 기여는 이론 수용장 전체에 고르게 퍼지지 않고 **중앙에 몰린 종 모양**이다.
> (b) 명목 범위(선형)와 실질 범위($\sqrt{L}$)의 격차가 깊이와 함께 벌어진다 — §11.2.3.
> (c) 같은 팽창률의 반복은 나머지류 하나에 갇힌다 — 서로소 관행의 근거 (§11.2.6).
> (d) 유효 수용장은 구조만의 함수가 아니다. 과제가 요구하면 학습이 기여를 옮긴다.
> 다만 **옮길 수 있는 범위는 이론 수용장 안**이라는 것 — 그것이 구조가 정하는 상한이다 (§11.2.7).

---
## 6. 자기 점검

1. (a)의 점선(가우시안)이 꼬리에서 실측과 어긋난다. 어느 쪽이 더 두꺼운가? 이유를 §11.2.3의 무작위 행보 논증으로 설명하라.
2. ReLU를 빼면(`relu=False`) (a)의 모양이 어떻게 되는가? 예측한 뒤 확인하라.
3. (d)에서 팽창을 [1,1,1,1]로 바꾸면 과제 정확도는 어떻게 되는가? 이론 수용장을 먼저 계산해 보라. 또, 과제를 두 값의 \'곱\'의 부호로 바꾸면 무엇이 더 필요해지는가?
4. 2차원에서 유효 수용장의 넓이는 깊이에 어떤 속도로 자라겠는가?

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `DEPTHS`, `N_INIT` | 2절 | — | 훑는 깊이와 평균 낼 초기화 수 |
| `sched_a/b` | 3절 | [1,2,4]/[2,2,2] | 팽창 스케줄. gcd가 1인지 확인 |
| `DELTA` | 4절 | 12 | 장거리 과제의 거리. 이론 수용장 밖으로 내보내 보라 |
| `STEPS` | 4절 | 400 | 학습 걸음 수 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")